# Qwen3 Approach–Refusal Holdout Audit

approachが既知のrefusal/compliance系方向に近い可能性を監査します。層ごとのcosineを測り、refusal proxy成分を除いたapproachが完全未使用の20個の無害promptでも残るか、100 random・新seedで評価します。

安全上、refusal方向を弱めて有害回答を生成する実験は行いません。harmful promptは活性取得だけに使い、生成評価は無害holdoutだけです。Colabでは **T4 GPU** を選択してください。

In [ ]:
!nvidia-smi
!pip -q install -U "transformers>=4.53,<6" accelerate pandas matplotlib

`colab_neurostate_3axis.py` と `colab_approach_refusal_holdout_audit.py` の2ファイルをアップロードしてください。

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
for stem in ('colab_neurostate_3axis','colab_approach_refusal_holdout_audit'):
    candidates=[name for name in uploaded if name.startswith(stem) and name.endswith('.py')]
    assert candidates,f'{stem}.py がありません'
    Path(stem+'.py').write_bytes(uploaded[candidates[-1]])
print('files ready')

## 本番監査
Qwen3 1.7B、source 18→target 20、未使用20 prompts、100 randomで実行します。

In [ ]:
!python colab_approach_refusal_holdout_audit.py --random-count 100 --random-seed 20260723 --output-dir qwen3_approach_refusal_holdout

In [ ]:
import json,pandas as pd,matplotlib.pyplot as plt
from pathlib import Path
summary=json.loads(Path('qwen3_approach_refusal_holdout/summary.json').read_text())
display(pd.DataFrame(summary['results']).T[['mean_slope','positive_prompts','random_abs_beating','rank_p','max_random_abs']])
print('selected cosine:',summary['selected_layer_cosine'])
print('approach retained after removal:',summary['approach_norm_retained_after_refusal_removal'])
plt.figure(figsize=(8,3)); plt.plot(summary['layerwise_approach_refusal_cosine'],marker='o'); plt.axhline(0,color='gray',linewidth=1); plt.xlabel('source layer'); plt.ylabel('cosine'); plt.title('Approach vs harmful–harmless refusal proxy'); plt.show()

`approach_without_refusal`が20/20に近い一貫性と小さいrank pを維持すれば、approachは単なるrefusal proxyの言い換えではない証拠が強まります。refusal proxyはArditi et al.の完全再現ではなく、harmful-minus-harmless活性差による監査用近似です。

In [ ]:
!zip -qr qwen3_approach_refusal_holdout_results.zip qwen3_approach_refusal_holdout
from google.colab import files
files.download('qwen3_approach_refusal_holdout_results.zip')